movies.csv'de description ve tagline çıkarıldı ve eğer name,date,minute,ratingde herhangi bir null varsa o satır da çıkarıldı.

In [ ]:
import pandas as pd

df_movies = pd.read_csv("movies.csv")

df_movies = df_movies.drop(columns=["tagline", "description"])

df_movies = df_movies.dropna(
    subset=["name", "date", "minute", "rating"]
)
df_movies

,id,name,date,minute,rating
0,1000001,Barbie,2023.0,114.0,3.86
1,1000002,Parasite,2019.0,133.0,4.56
2,1000003,Everything Everywhere All at Once,2022.0,140.0,4.30
3,1000004,Fight Club,1999.0,139.0,4.27
4,1000005,La La Land,2016.0,129.0,4.09
...,...,...,...,...,...
164282,1164283,Naughty Grandma 2,2019.0,86.0,2.66
164848,1164849,To Steal a Thief,1996.0,93.0,2.95
165146,1165147,9 millimeter,1997.0,96.0,2.77
165759,1165760,Dreamy,2013.0,90.0,2.28


In [ ]:
print(df_movies[["id", "name", "date", "minute", "rating"]].isna().sum())

id        0
name      0
date      0
minute    0
rating    0
dtype: int64


languages.csv

In [ ]:
import pandas as pd

df_languages = pd.read_csv("languages.csv")

top_languages = [
    "English", "French", "Spanish", "German", "Japanese",
    "Portuguese", "Chinese", "Russian", "Italian", "Korean",
    "Arabic", "Hindi", "Swedish", "Dutch", "Tagalog",
    "Czech", "Turkish", "Polish", "Cantonese", "Danish"
]

mapping = {
    "Language": 2,
    "Primary language": 2,
    "Spoken language": 1
}

# Değerleri dönüştür
df_languages["value"] = df_languages["type"].map(mapping)

# Top 20 dışında kalan dillere "Other" adını ver
df_languages["language"] = df_languages["language"].apply(
    lambda x: x if x in top_languages else "Other"
)

# One-Hot / 0-1-2 encoding
languages_encoded = (
    df_languages
    .pivot_table(
        index="id",
        columns="language",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

print(languages_encoded.head())
print(languages_encoded.shape)

language       id  Arabic  Cantonese  Chinese  Czech  Danish  Dutch  English  \
0         1000001       0          0        0      0       0      0        2   
1         1000002       0          0        0      0       0      0        1   
2         1000003       0          1        1      0       0      0        2   
3         1000004       0          0        0      0       0      0        2   
4         1000005       0          0        0      0       0      0        2   

language  French  German  ...  Japanese  Korean  Other  Polish  Portuguese  \
0              0       0  ...         0       0      0       0           0   
1              0       1  ...         0       2      0       0           0   
2              0       0  ...         0       0      0       0           0   
3              0       0  ...         0       0      0       0           0   
4              0       0  ...         0       0      0       0           0   

language  Russian  Spanish  Swedish  Tagalog  Turk

Left join

In [ ]:
df_final = df_movies.merge(
    languages_encoded,
    on="id",
    how="left"
)

1. Her dil sütununda kaç tane NaN var?

In [ ]:
language_columns = top_languages + ["Other"]

print(df_final[language_columns].isna().sum())

English       3196
French        3196
Spanish       3196
German        3196
Japanese      3196
Portuguese    3196
Chinese       3196
Russian       3196
Italian       3196
Korean        3196
Arabic        3196
Hindi         3196
Swedish       3196
Dutch         3196
Tagalog       3196
Czech         3196
Turkish       3196
Polish        3196
Cantonese     3196
Danish        3196
Other         3196
dtype: int64


2. Tüm dil sütunları 0 olan film var mı?

In [ ]:
language_columns = top_languages + ["Other"]

mask = df_final[language_columns].fillna(0).eq(0).all(axis=1)

print("Tüm dil sütunları 0 olan film sayısı:", mask.sum())

print(df_final.loc[mask, ["id", "name", "date"] + language_columns].head(20))

Tüm dil sütunları 0 olan film sayısı: 3196
           id                            name    date  English  French  \
870   1000871                      Metropolis  1927.0      NaN     NaN   
960   1000961                       Nosferatu  1922.0      NaN     NaN   
1120  1001121              A Trip to the Moon  1902.0      NaN     NaN   
1156  1001157     The Cabinet of Dr. Caligari  1920.0      NaN     NaN   
1444  1001445                     City Lights  1931.0      NaN     NaN   
1512  1001513      The Passion of Joan of Arc  1928.0      NaN     NaN   
1546  1001547                         Mad God  2021.0      NaN     NaN   
1646  1001647                      The Artist  2011.0      NaN     NaN   
1714  1001715                    Robot Dreams  2023.0      NaN     NaN   
1885  1001886                Un Chien Andalou  1929.0      NaN     NaN   
2112  1002113                   Sherlock, Jr.  1924.0      NaN     NaN   
2128  1002129         Man with a Movie Camera  1929.0      NaN     Na

tüm dil sütunları nan olan filmleri silmek

In [ ]:
language_columns = top_languages + ["Other"]

mask = df_final[language_columns].fillna(0).eq(0).all(axis=1)

print("Silinen film sayısı:", mask.sum())

df_final = df_final[~mask].reset_index(drop=True)

Silinen film sayısı: 3196


genres.csv

In [ ]:
import pandas as pd

df_genres = pd.read_csv("genres.csv")

print(df_genres["genre"].unique())

['Comedy' 'Adventure' 'Thriller' 'Drama' 'Science Fiction' 'Action'
 'Music' 'Romance' 'History' 'Crime' 'Animation' 'Mystery' 'Horror'
 'Family' 'Fantasy' 'War' 'Western' 'TV Movie' 'Documentary']


In [ ]:
import pandas as pd

df_genres = pd.read_csv("genres.csv")

genres_encoded = (
    pd.crosstab(
        df_genres["id"],
        df_genres["genre"]
    )
    .reset_index()
)

print(genres_encoded.head())

print(genres_encoded.shape)

genre       id  Action  Adventure  Animation  Comedy  Crime  Documentary  \
0      1000001       0          1          0       1      0            0   
1      1000002       0          0          0       1      0            0   
2      1000003       1          1          0       1      0            0   
3      1000004       0          0          0       0      0            0   
4      1000005       0          0          0       1      0            0   

genre  Drama  Family  Fantasy  History  Horror  Music  Mystery  Romance  \
0          0       0        0        0       0      0        0        0   
1          1       0        0        0       0      0        0        0   
2          0       0        0        0       0      0        0        0   
3          1       0        0        0       0      0        0        0   
4          1       0        0        0       0      1        0        1   

genre  Science Fiction  TV Movie  Thriller  War  Western  
0                    0         0 

1. Left Join

In [ ]:
df_final = df_final.merge(
    genres_encoded,
    on="id",
    how="left"
)

2. Tür sütunlarını seç

In [ ]:
genre_columns = [col for col in genres_encoded.columns if col != "id"]

3. Hiç türü olmayan filmleri kontrol et

In [ ]:
mask = df_final[genre_columns].fillna(0).eq(0).all(axis=1)

print("Hiç genre bilgisi olmayan film sayısı:", mask.sum())

print(df_final.loc[mask, ["id", "name", "date"]].head(20))

Hiç genre bilgisi olmayan film sayısı: 1131
            id                                              name    date
1497   1001503                                    Obi-Wan Kenobi  2022.0
4716   1004760                                            Ahsoka  2023.0
5041   1005089                             The Book of Boba Fett  2021.0
6261   1006341                Love, Death & Robots: Good Hunting  2019.0
6367   1006447                     Love, Death & Robots: Ice Age  2019.0
6819   1006907                 BoJack Horseman Christmas Special  2014.0
6831   1006919                  Love, Death & Robots: Fish Night  2019.0
7054   1007146  Love, Death & Robots: Automated Customer Service  2021.0
7081   1007173                Love, Death & Robots: Helping Hand  2019.0
7291   1007388      Love, Death & Robots: Night of the Mini Dead  2022.0
7607   1007713                    Love, Death & Robots: Lucky 13  2019.0
7716   1007826                Love, Death & Robots: Mason's Rats  2022.0
7849   

4. Eğer silmeye karar verirsen

In [ ]:
genre_columns = [col for col in genres_encoded.columns if col != "id"]

mask = df_final[genre_columns].fillna(0).eq(0).all(axis=1)

print("Silinen film sayısı:", mask.sum())

df_final = df_final[~mask].reset_index(drop=True)

Silinen film sayısı: 1131


THEMES.CSV


In [ ]:
import pandas as pd



df_themes = pd.read_csv("themes.csv")



print(df_themes["id"].nunique())

24508


In [ ]:
import pandas as pd

df_themes = pd.read_csv("themes.csv")

# Theme sütunundaki tüm benzersiz değerleri al ve alfabetik sırala
unique_themes = sorted(df_themes["theme"].dropna().unique())

print(f"Toplam benzersiz tema sayısı: {len(unique_themes)}\n")

for theme in unique_themes:
    print(theme)

Toplam benzersiz tema sayısı: 109

Action comedy and silly heroics
Action-packed space and alien sagas
Adorable animals and heartwarming families
Adrenaline-fueled action and fast cars
Air pilot heroism and survival
Amusing jokes and witty satire
Bloody vampire horror
Bollywood emotional dramas
Bravery in War
Brutal, violent prison drama
Captivating relationships and charming romance
Captivating vision and Shakespearean drama
Catchy songs and hilarious musical comedy
Challenging or sexual themes & twists
Charming romances and delightful chemistry
Chilling experiments and classic monster horror
Creepy, chilling, and terrifying horror
Crime, drugs and gangsters
Crude humor and satire
Dance rhythms and catchy tunes
Dangerous technology and the apocalypse
Dazzling vocal performances and musicals
Disastrous voyages and heroic survival
Dreamlike, quirky, and surreal storytelling
Emotional LGBTQ relationships
Emotional and captivating fantasy storytelling
Emotional and touching family dramas


In [ ]:
import pandas as pd

df_themes = pd.read_csv("themes.csv")

# Tema eşleme sözlüğü
theme_mapping = {

    # ACTION
    "Action comedy and silly heroics": "Action_Theme",
    "Adrenaline-fueled action and fast cars": "Action_Theme",
    "Explosive and action-packed heroes vs. villains": "Action_Theme",
    "Graphic violence and brutal revenge": "Action_Theme",
    "Heists and thrilling action": "Action_Theme",
    "High speed and special ops": "Action_Theme",
    "Intense combat and martial arts": "Action_Theme",
    "Violent action, guns, and crime": "Action_Theme",

    # ADVENTURE
    "Epic adventure and breathtaking battles": "Adventure_Theme",
    "Fantasy adventure, heroism, and swordplay": "Adventure_Theme",
    "Kids' animated fun and adventure": "Adventure_Theme",
    "War and historical adventure": "Adventure_Theme",

    # SCI-FI
    "Action-packed space and alien sagas": "Sci-Fi",
    "Dangerous technology and the apocalypse": "Sci-Fi",
    "Humanity's odyssey: earth and beyond": "Sci-Fi",
    "Imaginative space odysseys and alien encounters": "Sci-Fi",
    "Monsters, aliens, sci-fi and the apocalypse": "Sci-Fi",
    "Thought-provoking sci-fi action and future technology": "Sci-Fi",

    # FANTASY
    "Emotional and captivating fantasy storytelling": "Fantasy_Theme",
    "Fairy-tale fantasy and enchanted magic": "Fantasy_Theme",

    # HORROR
    "Bloody vampire horror": "Horror_Theme",
    "Chilling experiments and classic monster horror": "Horror_Theme",
    "Creepy, chilling, and terrifying horror": "Horror_Theme",
    "Extreme gory horror and cannibalism": "Horror_Theme",
    "Gory, gruesome, and slasher horror": "Horror_Theme",
    "Gothic and eerie haunting horror": "Horror_Theme",
    "Horror, the undead and monster classics": "Horror_Theme",
    "Sci-fi horror, creatures, and aliens": "Horror_Theme",
    "Survival horror and zombie carnage": "Horror_Theme",
    "Terrifying, haunted, and supernatural horror": "Horror_Theme",

    # CRIME
    "Crime, drugs and gangsters": "Crime_Theme",
    "Engaging, intense crime and casino drama": "Crime_Theme",
    "Gripping, intense violent crime": "Crime_Theme",
    "Gritty crime and ruthless gangsters": "Crime_Theme",
    "Noir and dark crime dramas": "Crime_Theme",
    "Suspenseful crime thrillers": "Crime_Theme",
    "Violent crime and drugs": "Crime_Theme",

    # THRILLER
    "Exciting spy thrillers with tense intrigue": "Thriller_Theme",
    "Intense political and terrorist thrillers": "Thriller_Theme",
    "Intriguing and suspenseful murder mysteries": "Thriller_Theme",
    "Thrillers and murder mysteries": "Thriller_Theme",
    "Twisted dark psychological thriller": "Thriller_Theme",

    # DRAMA
    "Brutal, violent prison drama": "Drama_Theme",
    "Captivating vision and Shakespearean drama": "Drama_Theme",
    "Powerful poetic and passionate drama": "Drama_Theme",
    "Powerful stories of heartbreak and suffering": "Drama_Theme",
    "Tragic sadness and captivating beauty": "Drama_Theme",

    # FAMILY
    "Adorable animals and heartwarming families": "Family_Theme",
    "Emotional and touching family dramas": "Family_Theme",
    "Enduring stories of family and marital drama": "Family_Theme",
    "Heartbreaking and moving family drama": "Family_Theme",
    "Touching and sentimental family stories": "Family_Theme",

    # ROMANCE
    "Captivating relationships and charming romance": "Romance_Theme",
    "Charming romances and delightful chemistry": "Romance_Theme",
    "Erotic relationships and desire": "Romance_Theme",
    "Laugh-out-loud relationship entanglements": "Romance_Theme",
    "Moving relationship stories": "Romance_Theme",
    "Passion and romance": "Romance_Theme",
    "Quirky and endearing relationships": "Romance_Theme",

    # COMEDY
    "Amusing jokes and witty satire": "Comedy_Theme",
    "Crude humor and satire": "Comedy_Theme",
    "Funny jokes and crude humor": "Comedy_Theme",
    "Gags, jokes, and slapstick humor": "Comedy_Theme",
    "Relationship comedy": "Comedy_Theme",
    "Spooky, scary comedy": "Comedy_Theme",
    "Teen school antics and laughter": "Comedy_Theme",

    # MUSICAL
    "Catchy songs and hilarious musical comedy": "Musical",
    "Dance rhythms and catchy tunes": "Musical",
    "Dazzling vocal performances and musicals": "Musical",
    "Legendary musicians and stardom": "Musical",
    "Song and dance": "Musical",

    # COMING OF AGE
    "Emotional teen coming-of-age stories": "Coming_of_Age",
    "Student coming-of-age challenges": "Coming_of_Age",
    "Teen friendship and coming-of-age": "Coming_of_Age",
    "Underdogs and coming of age": "Coming_of_Age",

    # SPORTS
    "Inspiring sports underdog stories": "Sports",
    "Underdog fighting and boxing stories": "Sports",

    # WAR
    "Air pilot heroism and survival": "War_Theme",
    "Bravery in War": "War_Theme",
    "Historical battles and epic heroism": "War_Theme",
    "Military combat and heroic soldiers": "War_Theme",
    "Nazis and World War II": "War_Theme",
    "Political drama, patriotism, and war": "War_Theme",

    # HISTORY
    "Epic history and literature": "History_Theme",
    "Lavish dramas and sumptuous royalty": "History_Theme",

    # POLITICS
    "Politics and human rights": "Politics",
    "Politics, propaganda, and political documentaries": "Politics",
    "Riveting political and presidential drama": "Politics",

    # RELIGION
    "Faith and religion": "Religion",
    "Faith and spiritual journeys": "Religion",
    "Religious faith, sin, and forgiveness": "Religion",

    # DOCUMENTARY
    "Fascinating, emotional stories and documentaries": "Documentary_Theme",

    # BIOGRAPHY
    "Bollywood emotional dramas": "Biography",
    "Emotional life of renowned artists": "Biography",

    # SOCIAL ISSUES
    "Challenging or sexual themes & twists": "Social_Issues",
    "Emotional LGBTQ relationships": "Social_Issues",
    "Intense violence and sexual transgression": "Social_Issues",
    "Racism and the powerful fight for justice": "Social_Issues",

    # HUMANITY
    "Humanity and the world around us": "Humanity",

    # SURVIVAL
    "Disastrous voyages and heroic survival": "Survival",

    # SURREAL
    "Dreamlike, quirky, and surreal storytelling": "Surreal",
    "Surreal and thought-provoking visions of life and death": "Surreal",

    # SUPERHERO
    "Epic heroes": "Superhero",
    "Superheroes in action-packed battles with villains": "Superhero",

    # MONSTERS
    "Sci-fi monster and dinosaur adventures": "Monsters",

    # HOLIDAY
    "Holiday joy and heartwarming Christmas": "Holiday",

    # WESTERN
    "Westerns": "Western_Theme",
    "Western frontier dramas with a touch of humor": "Western_Theme",
    "Wild west outlaws and gunfights": "Western_Theme",
}

# Mapping'de olmayan temaları çıkar
df_themes = df_themes[df_themes["theme"].isin(theme_mapping.keys())].copy()

# Ana kategorilere dönüştür
df_themes["theme"] = df_themes["theme"].map(theme_mapping)

# One-hot encoding
themes_encoded = (
    pd.crosstab(
        df_themes["id"],
        df_themes["theme"]
    )
    .reset_index()
)

print(themes_encoded.head())
print(themes_encoded.shape)
print(themes_encoded.columns.tolist())

theme       id  Action_Theme  Adventure_Theme  Biography  Comedy_Theme  \
0      1000001             0                0          0             2   
1      1000002             0                0          0             0   
2      1000003             0                0          0             1   
3      1000004             1                0          0             0   
4      1000005             0                0          1             0   

theme  Coming_of_Age  Crime_Theme  Documentary_Theme  Drama_Theme  \
0                  0            0                  0            0   
1                  0            0                  0            0   
2                  0            0                  0            1   
3                  0            0                  0            0   
4                  0            0                  0            0   

theme  Family_Theme  ...  Romance_Theme  Sci-Fi  Social_Issues  Sports  \
0                 0  ...              3       0              0    

1. Left Join

In [ ]:
df_final = df_final.merge(
    themes_encoded,
    on="id",
    how="left"
)

2. Theme sütunlarını seç ve hepsi 0 olanları bul

In [ ]:
theme_columns = [col for col in themes_encoded.columns if col != "id"]

df_final[theme_columns] = df_final[theme_columns].fillna(0)

print("Her theme sütunundaki NaN sayısı:")
print(df_final[theme_columns].isna().sum())

Her theme sütunundaki NaN sayısı:
Action_Theme         0
Adventure_Theme      0
Biography            0
Comedy_Theme         0
Coming_of_Age        0
Crime_Theme          0
Documentary_Theme    0
Drama_Theme          0
Family_Theme         0
Fantasy_Theme        0
History_Theme        0
Holiday              0
Horror_Theme         0
Humanity             0
Monsters             0
Musical              0
Politics             0
Religion             0
Romance_Theme        0
Sci-Fi               0
Social_Issues        0
Sports               0
Superhero            0
Surreal              0
Survival             0
Thriller_Theme       0
War_Theme            0
Western_Theme        0
dtype: int64


In [ ]:
print("Tüm theme sütunları 0 olan film sayısı:",
      df_final[theme_columns].eq(0).all(axis=1).sum())

Tüm theme sütunları 0 olan film sayısı: 62825


**STUDIOS.CSV**

In [ ]:
import pandas as pd

studios = pd.read_csv('studios.csv')



studios_features = studios.groupby('id').agg(
n_studios=('studio', 'nunique')
).reset_index()



major_studios = [
'Warner Bros. Pictures', 'Universal Pictures', 'Paramount',
'Columbia Pictures', 'Metro-Goldwyn-Mayer', 'Walt Disney Pictures',
'20th Century Fox', 'Sony Pictures'
]

studios['is_major'] = studios['studio'].isin(major_studios)
major_flag = studios.groupby('id')['is_major'].any().astype(int).reset_index()
major_flag = major_flag.rename(columns={'is_major': 'has_major_studio'})

studios_features = studios_features.merge(major_flag, on='id', how='left')

# stüdyo frekans encoding (o stüdyonun toplam filmografideki sıklığı -> "büyüklük" proxy'si)
studio_freq = studios['studio'].value_counts()
studios['studio_freq'] = studios['studio'].map(studio_freq)
avg_studio_freq = studios.groupby('id')['studio_freq'].max().reset_index()
avg_studio_freq = avg_studio_freq.rename(columns={'studio_freq': 'max_studio_frequency'})

studios_features = studios_features.merge(avg_studio_freq, on='id', how='left')

print(studios_features.shape)
studios_features.head()

(438197, 4)


,id,n_studios,has_major_studio,max_studio_frequency
0,1000001,5,1,3222.0
1,1000002,1,0,32.0
2,1000003,4,0,24.0
3,1000004,5,1,1693.0
4,1000005,6,0,146.0


Önce merge yapalım:

In [ ]:
df_final = df_final.merge(
    studios_features,
    on="id",
    how="left"
)

Sonra kontrol:

In [ ]:
studio_columns = [
    "n_studios",
    "has_major_studio",
    "max_studio_frequency"
]

print(df_final[studio_columns].isna().sum())

print(df_final[df_final[studio_columns].isna().any(axis=1)][
    ["id", "name"] + studio_columns
].head(20))

n_studios               9902
has_major_studio        9902
max_studio_frequency    9902
dtype: int64
           id                                               name  n_studios  \
568   1000569                                 Bo Burnham: Inside        NaN   
1717  1001728                        Black Mirror: Joan Is Awful        NaN   
2024  1002036         Euphoria: F*ck Anyone Who’s Not a Sea Blob        NaN   
2138  1002153                           Black Mirror: Loch Henry        NaN   
2197  1002214                       Black Mirror: Beyond the Sea        NaN   
2371  1002390                             Black Mirror: Nosedive        NaN   
2459  1002479                    Black Mirror: Shut Up and Dance        NaN   
2675  1002699                          Black Mirror: Hang the DJ        NaN   
2785  1002811                        Black Mirror: USS Callister        NaN   
2796  1002822               Black Mirror: Fifteen Million Merits        NaN   
2801  1002827                  

In [ ]:
print("Silinen film sayısı:", df_final["n_studios"].isna().sum())

df_final = df_final.dropna(subset=["n_studios"]).reset_index(drop=True)

Silinen film sayısı: 9902


RELEASES.CSV

In [ ]:
import pandas as pd



df_releases = pd.read_csv("releases.csv")



print(df_releases["country"].nunique())

246


In [ ]:
import pandas as pd

df_releases = pd.read_csv("releases.csv")

country_counts = (
    df_releases[["id", "country"]]
    .drop_duplicates()
    .groupby("country")
    .size()
    .sort_values(ascending=False)
)

print(country_counts)

country
USA                                  290006
France                                81092
Germany                               80147
UK                                    75393
Japan                                 45123
                                      ...  
Nauru                                    56
Heard Island and McDonald Islands        55
Svalbard and Jan Mayen                   47
French Southern Territories              39
Netherlands Antilles                     23
Length: 246, dtype: int64


In [ ]:
import pandas as pd

df_releases = pd.read_csv("releases.csv")

country_counts = (
    df_releases[["id", "country"]]
    .drop_duplicates()
    .groupby("country")
    .size()
    .sort_values(ascending=False)
)

for country, count in country_counts.head(25).items():
    print(f"{country}: {count}")

USA: 290006
France: 81092
Germany: 80147
UK: 75393
Japan: 45123
Canada: 34885
Brazil: 34675
Italy: 34066
Spain: 30404
India: 30220
Russian Federation: 23057
South Korea: 22976
Netherlands: 22433
Mexico: 21264
Australia: 20988
China: 17437
Sweden: 17295
Argentina: 13620
Denmark: 13194
Portugal: 12813
Greece: 12791
Hong Kong: 12214
Poland: 11828
Philippines: 10777
Turkey: 10773


In [ ]:
import pandas as pd

releases = pd.read_csv("releases.csv")

releases = releases.drop(columns=["rating"])
releases["date"] = pd.to_datetime(releases["date"], errors="coerce")

# --- id bazında temel özellikler ---
releases_features = releases.groupby("id").agg(
    n_countries=("country", "nunique"),
    n_release_types=("type", "nunique"),
    first_release_date=("date", "min"),
    last_release_date=("date", "max")
).reset_index()

# --- Release type flag'leri ---
type_dummies = pd.crosstab(releases["id"], releases["type"])
type_dummies = (type_dummies > 0).astype(int)
type_dummies.columns = [
    f'has_{c.lower().replace(" ", "_")}'
    for c in type_dummies.columns
]
type_dummies = type_dummies.reset_index()

releases_features = releases_features.merge(
    type_dummies,
    on="id",
    how="left"
)

# --- İlk çıktığı ülke ---
releases_sorted = releases.sort_values("date")

first_country = (
    releases_sorted
    .dropna(subset=["date"])
    .groupby("id")["country"]
    .first()
    .rename("first_release_country")
    .reset_index()
)

releases_features = releases_features.merge(
    first_country,
    on="id",
    how="left"
)

# --- Sadece en çok kullanılan 25 ülke için one-hot encoding ---
top_countries = [
    "USA",
    "France",
    "Germany",
    "UK",
    "Japan",
    "Canada",
    "Brazil",
    "Italy",
    "Spain",
    "India",
    "Russian Federation",
    "South Korea",
    "Netherlands",
    "Mexico",
    "Australia",
    "China",
    "Sweden",
    "Argentina",
    "Denmark",
    "Portugal",
    "Greece",
    "Hong Kong",
    "Poland",
    "Philippines",
    "Turkey"
]

country_dummies = pd.crosstab(
    releases["id"],
    releases["country"]
)

country_dummies = country_dummies.reindex(
    columns=top_countries,
    fill_value=0
)

country_dummies = (country_dummies > 0).astype(int)

country_dummies.columns = [
    f"released_in_{c.lower().replace(' ', '_').replace(',', '').replace('(', '').replace(')', '')}"
    for c in country_dummies.columns
]

country_dummies = country_dummies.reset_index()

releases_features = releases_features.merge(
    country_dummies,
    on="id",
    how="left"
)

country_cols = [
    c for c in releases_features.columns
    if c.startswith("released_in_")
]

releases_features[country_cols] = (
    releases_features[country_cols]
    .fillna(0)
    .astype(int)
)

# --- Tarihten yeni özellikler ---
releases_features["release_year"] = (
    releases_features["first_release_date"].dt.year
)

releases_features["release_month"] = (
    releases_features["first_release_date"].dt.month
)

releases_features["release_quarter"] = (
    releases_features["first_release_date"].dt.quarter
)

# --- Theatrical -> Digital geçiş süresi ---
theatrical_dates = (
    releases[releases["type"] == "Theatrical"]
    .groupby("id")["date"]
    .min()
)

digital_dates = (
    releases[releases["type"] == "Digital"]
    .groupby("id")["date"]
    .min()
)

gap = (digital_dates - theatrical_dates).dt.days

releases_features = releases_features.merge(
    gap.rename("days_theatrical_to_digital"),
    on="id",
    how="left"
)

print(releases_features.shape)
print(releases_features.head())

(826018, 41)
        id  n_countries  n_release_types first_release_date last_release_date  \
0  1000001           58                6         2023-07-06        2023-12-16   
1  1000002           62                6         2019-05-21        2022-07-01   
2  1000003           47                6         2022-03-11        2023-06-13   
3  1000004           51                5         1999-09-10        2024-03-15   
4  1000005           78                6         2016-08-31        2024-12-03   

   has_digital  has_physical  has_premiere  has_tv  has_theatrical  ...  \
0            1             1             1       1               1  ...   
1            1             1             1       1               1  ...   
2            1             1             1       1               1  ...   
3            1             1             1       1               1  ...   
4            1             1             1       1               1  ...   

   released_in_portugal released_in_greece  relea

2. Yeni gelen sütunlarda kaç NaN var?

In [ ]:
# Release sütunları (id ve kullanılmayacak sütun hariç)
release_columns = [
    col for col in releases_features.columns
    if col not in ["id", "days_theatrical_to_digital"]
]

print("Release sütunları:")
print(release_columns)

# Eski release sütunlarını sil
df_final = df_final.drop(
    columns=[c for c in release_columns if c in df_final.columns],
    errors="ignore"
)

print("\nEski release sütunları silindi.")

# Release verisini tekrar ekle
df_final = df_final.merge(
    releases_features.drop(columns=["days_theatrical_to_digital"], errors="ignore"),
    on="id",
    how="left"
)

print("Merge tamamlandı.")

# Merge sonrası oluşan NaN'ları 0 yap
df_final[release_columns] = df_final[release_columns].fillna(0)

print("NaN değerleri 0 yapıldı.")

print("\nDataFrame boyutu:")
print(df_final.shape)

print("\nRelease sütunlarındaki kalan NaN sayıları:")
print(df_final[release_columns].isna().sum())

print("\nİlk 5 satır:")
print(df_final[["id"] + release_columns].head())

Release sütunları:
['n_countries', 'n_release_types', 'first_release_date', 'last_release_date', 'has_digital', 'has_physical', 'has_premiere', 'has_tv', 'has_theatrical', 'has_theatrical_limited', 'first_release_country', 'released_in_usa', 'released_in_france', 'released_in_germany', 'released_in_uk', 'released_in_japan', 'released_in_canada', 'released_in_brazil', 'released_in_italy', 'released_in_spain', 'released_in_india', 'released_in_russian_federation', 'released_in_south_korea', 'released_in_netherlands', 'released_in_mexico', 'released_in_australia', 'released_in_china', 'released_in_sweden', 'released_in_argentina', 'released_in_denmark', 'released_in_portugal', 'released_in_greece', 'released_in_hong_kong', 'released_in_poland', 'released_in_philippines', 'released_in_turkey', 'release_year', 'release_month', 'release_quarter']

Eski release sütunları silindi.
Merge tamamlandı.
NaN değerleri 0 yapıldı.

DataFrame boyutu:
(76246, 115)

Release sütunlarındaki kalan NaN sayıl

CREW.CSV

In [ ]:
import pandas as pd

df_crew = pd.read_csv("crew.csv")

roles = [
    "Director",
    "Writer",
    "Producer",
    "Composer",
    "Editor",
    "Cinematography"
]

crew_counts = (
    df_crew[df_crew["role"].isin(roles)]
    .groupby(["id", "role"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

crew_counts = crew_counts.rename(columns={
    "Director": "director_count",
    "Writer": "writer_count",
    "Producer": "producer_count",
    "Composer": "composer_count",
    "Editor": "editor_count",
    "Cinematography": "cinematography_count"
})

# -----------------------------
# Top director feature
# -----------------------------

directors = [
    "Akira Kurosawa",
    "Ingmar Bergman",
    "Christopher Nolan",
    "Andrei Tarkovsky",
    "Martin Scorsese",
    "Hayao Miyazaki",
    "Stanley Kubrick",
    "Alfred Hitchcock",
    "Billy Wilder",
    "Charlie Chaplin",
    "Hirokazu Kore-eda",
    "Quentin Tarantino",
    "Abbas Kiarostami",
    "Masaki Kobayashi",
    "Denis Villeneuve",
    "David Lynch",
    "Paul Thomas Anderson",
    "Francis Ford Coppola",
    "Wong Kar-Wai",
    "Wim Wenders",
    "Sergio Leone",
    "Sidney Lumet",
    "Federico Fellini",
    "Robert Altman",
    "Yasujirō Ozu",
    "Steven Spielberg",
    "Ridley Scott",
    "Bong Joon Ho",
    "Peter Jackson",
    "Park Chan-wook",
    "Alfonso Cuarón",
    "Roman Polanski",
    "Hideaki Anno",
    "Lars von Trier",
    "Satoshi Kon",
    "Agnès Varda",
    "John Cassavetes",
    "Frank Capra",
    "William Wyler",
    "Michael Powell"
]

top_director = (
    df_crew[
        (df_crew["role"] == "Director") &
        (df_crew["name"].isin(directors))
    ][["id"]]
    .drop_duplicates()
)

top_director["top50_director"] = 1

crew_counts = crew_counts.merge(
    top_director,
    on="id",
    how="left"
)

crew_counts["top50_director"] = (
    crew_counts["top50_director"]
    .fillna(0)
    .astype(int)
)

print(crew_counts.head())

        id  cinematography_count  composer_count  director_count  \
0  1000001                     1               2               1   
1  1000002                     1               1               1   
2  1000003                     1               3               2   
3  1000004                     1               2               1   
4  1000005                     1               1               1   

   editor_count  producer_count  writer_count  top50_director  
0             1               5             2               0  
1             1               6             3               0  
2             1               9             2               0  
3             1               4             2               0  
4             1               5             1               0  


In [ ]:
# Crew sütunları
crew_columns = [
    "director_count",
    "writer_count",
    "producer_count",
    "composer_count",
    "editor_count",
    "cinematography_count",
    "top50_director"
]

# Daha önce merge edildiyse eski sütunları kaldır
df_final = df_final.drop(columns=crew_columns, errors="ignore")

# Crew verisini tekrar ekle
df_final = df_final.merge(
    crew_counts,
    on="id",
    how="left"
)

# Top50 director NaN ise 0 yap
df_final["top50_director"] = (
    df_final["top50_director"]
    .fillna(0)
    .astype(int)
)

# Diğer crew sütunlarında NaN var mı?
check_columns = [
    "director_count",
    "writer_count",
    "producer_count",
    "composer_count",
    "editor_count",
    "cinematography_count"
]

print(df_final[check_columns].isna().sum())

print(df_final.loc[
    df_final[check_columns].isna().any(axis=1),
    ["id", "name"] + check_columns
].head(20))

print("Silinecek film sayısı:",
      df_final[check_columns].isna().any(axis=1).sum())

# Crew bilgisi eksik olan filmleri sil
df_final = df_final.dropna(subset=check_columns).reset_index(drop=True)

print("Yeni boyut:", df_final.shape)

director_count          244
writer_count            244
producer_count          244
composer_count          244
editor_count            244
cinematography_count    244
dtype: int64
            id                                           name  director_count  \
12254  1012666              Andor: A Disney+ Day Special Look             NaN   
17132  1017886      Ted Lasso: The Missing Christmas Mustache             NaN   
17496  1018282                       The Office Retrospective             NaN   
19032  1019958                            Worst Roommate Ever             NaN   
19866  1020859                             Surviving R. Kelly             NaN   
20852  1021963              Money Heist: From Tokyo to Berlin             NaN   
21542  1022724                    The Way of the Househusband             NaN   
22187  1023434      Spider-Man: All Roads Lead to No Way Home             NaN   
23574  1024974           Taylor Swift: The Road to Reputation             NaN   
23921  10

In [ ]:
crew_columns = [
    "director_count",
    "writer_count",
    "producer_count",
    "composer_count",
    "editor_count",
    "cinematography_count",
    "top50_director"
]

# Var olan crew kolonlarını bul
check_columns = [col for col in crew_columns if col in df_final.columns]

# NaN olan satırları sil
df_final = df_final.dropna(
    subset=check_columns
).reset_index(drop=True)

print("Kontrol edilen kolonlar:", check_columns)
print("Yeni boyut:", df_final.shape)

Kontrol edilen kolonlar: []
Yeni boyut: (76246, 115)


COUNTRIES.CSV

In [ ]:
import pandas as pd

df_countries = pd.read_csv("countries.csv")

top_countries = [
    "USA",
    "France",
    "UK",
    "Japan",
    "Germany",
    "Canada",
    "India",
    "Italy",
    "Brazil",
    "Spain",
    "Mexico",
    "China",
    "Russian Federation",
    "South Korea",
    "USSR",
    "Argentina",
    "Sweden",
    "Australia",
    "Philippines",
    "Hong Kong"
]

countries_encoded = pd.crosstab(
    df_countries["id"],
    df_countries["country"]
)

# Sadece ilk 20 ülkeyi tut
countries_encoded = countries_encoded.reindex(
    columns=top_countries,
    fill_value=0
)

countries_encoded = countries_encoded.reset_index()

print(countries_encoded.head())
print(countries_encoded.shape)

country       id  USA  France  UK  Japan  Germany  Canada  India  Italy  \
0        1000001    1       0   1      0        0       0      0      0   
1        1000002    0       0   0      0        0       0      0      0   
2        1000003    1       0   0      0        0       0      0      0   
3        1000004    1       0   0      0        1       0      0      0   
4        1000005    1       0   0      0        0       0      0      0   

country  Brazil  ...  Mexico  China  Russian Federation  South Korea  USSR  \
0             0  ...       0      0                   0            0     0   
1             0  ...       0      0                   0            1     0   
2             0  ...       0      0                   0            0     0   
3             0  ...       0      0                   0            0     0   
4             0  ...       0      0                   0            0     0   

country  Argentina  Sweden  Australia  Philippines  Hong Kong  
0               

In [ ]:
df_final = df_final.merge(
    countries_encoded,
    on="id",
    how="left"
)

Yeni gelen sütunları kontrol et

In [ ]:
country_columns = [col for col in countries_encoded.columns if col != "id"]

print(df_final[country_columns].isna().sum())

print(df_final.loc[
    df_final[country_columns].isna().any(axis=1),
    ["id", "name"] + country_columns
].head(20))

USA                   719
France                719
UK                    719
Japan                 719
Germany               719
Canada                719
India                 719
Italy                 719
Brazil                719
Spain                 719
Mexico                719
China                 719
Russian Federation    719
South Korea           719
USSR                  719
Argentina             719
Sweden                719
Australia             719
Philippines           719
Hong Kong             719
dtype: int64
            id                                               name  USA  \
3818   1003871                                        Lovers Rock  NaN   
5516   1005629                                      Come Together  NaN   
7168   1007340  Love, Death & Robots: Three Robots: Exit Strat...  NaN   
9344   1009618                        Doctor Who: The End of Time  NaN   
9859   1010151                                       Nymphomaniac  NaN   
10924  1011263         

NaN değerleri 0 yapmak

In [ ]:
country_columns = [col for col in countries_encoded.columns if col != "id"]

df_final[country_columns] = df_final[country_columns].fillna(0)

print(df_final[country_columns].isna().sum())

USA                   0
France                0
UK                    0
Japan                 0
Germany               0
Canada                0
India                 0
Italy                 0
Brazil                0
Spain                 0
Mexico                0
China                 0
Russian Federation    0
South Korea           0
USSR                  0
Argentina             0
Sweden                0
Australia             0
Philippines           0
Hong Kong             0
dtype: int64


In [ ]:
print("Toplam NaN:",
      df_final[country_columns].isna().sum().sum())

Toplam NaN: 0


ACTORS.CSV

In [ ]:
import pandas as pd

df_actors = pd.read_csv("actors.csv")

# Oyuncu sayısı
actors_features = (
    df_actors.groupby("id")
    .agg(
        actor_count=("name", "count"),
        unique_role_count=("role", "nunique")
    )
    .reset_index()
)

# -----------------------------
# Top actor feature
# -----------------------------

top_actors = [
    "Henry Fonda",
    "John Cusack",
    "Alec Baldwin",
    "Christopher Plummer",
    "Dennis Quaid",
    "Ed Harris",
    "John Goodman",
    "Barbara Stanwyck",
    "Christopher Walken",
    "John Malkovich",
    "Meryl Streep",
    "Spencer Tracy",
    "Toshirō Mifune",
    "Burt Lancaster",
    "Donald Sutherland",
    "Humphrey Bogart",
    "James Franco",
    "James Mason",
    "Jeff Bridges",
    "Robert Duvall",
    "Robert Mitchum",
    "Gary Oldman",
    "Ewan McGregor",
    "Gary Cooper",
    "Helen Mirren",
    "Robert Downey Jr.",
    "Robin Williams",
    "Al Pacino",
    "Christopher Lee",
    "Glenn Ford",
    "Harvey Keitel",
    "Johnny Depp",
    "Mark Wahlberg",
    "Colin Firth",
    "Ralph Fiennes",
    "George Sanders",
    "Isabelle Huppert",
    "Matt Damon",
    "Nick Nolte",
    "Sylvester Stallone",
    "Toni Collette",
    "Cate Blanchett",
    "Clint Eastwood",
    "John Travolta",
    "Paul Newman",
    "Scarlett Johansson",
    "Steve Buscemi",
    "Adam Sandler",
    "Ben Affleck",
    "Bill Nighy",
    "Brad Pitt",
    "Colin Farrell",
    "Denzel Washington",
    "Dustin Hoffman",
    "Kirk Douglas",
    "Pierce Brosnan",
    "Sam Rockwell",
    "Sean Connery",
    "Catherine Deneuve",
    "Edward G. Robinson",
    "Harrison Ford",
    "John Mills",
    "Marcello Mastroianni",
    "Max von Sydow",
    "Tommy Lee Jones",
    "Danny Glover",
    "John Hurt",
    "Julia Roberts",
    "Laurence Fishburne",
    "Ray Milland",
    "Richard Gere",
    "Sam Neill",
    "Anthony Quinn",
    "Ben Kingsley",
    "Herbert Lom",
    "Jeff Goldblum",
    "Kevin Costner",
    "Robert Ryan",
    "Will Ferrell",
    "Alfred Molina",
    "Bill Murray",
    "Cary Grant",
    "Christian Bale",
    "Jack Nicholson",
    "Mel Gibson",
    "Michelle Pfeiffer",
    "Peter Cushing",
    "Sean Penn",
    "Sigourney Weaver",
    "Stellan Skarsgård",
    "Steve Zahn",
    "Tom Cruise",
    "Eddie Murphy",
    "Gregory Peck",
    "Guy Pearce",
    "Jennifer Jason Leigh",
    "Jude Law",
    "Owen Wilson",
    "Ryan Reynolds",
    "Shah Rukh Khan",
    "Alan Arkin",
    "Basil Rathbone",
    "Catherine Keener",
    "Charlize Theron",
    "Dan Aykroyd",
    "Danny DeVito",
    "Jeremy Irons",
    "Kevin Spacey",
    "Naomi Watts",
    "William Holden",
    "Dennis Hopper",
    "Diane Keaton",
    "Gerard Butler",
    "Jean-Claude Van Damme",
    "Joan Crawford",
    "John Leguizamo",
    "Kevin Bacon",
    "Kristen Stewart",
    "Matthew McConaughey",
    "Michael Keaton",
    "Paul Giamatti",
    "Richard Widmark",
    "Tilda Swinton",
    "Tom Wilkinson",
    "Forest Whitaker",
    "Fredric March",
    "J.K. Simmons",
    "Jake Gyllenhaal",
    "Jason Statham",
    "Jim Broadbent",
    "John Turturro",
    "Randolph Scott",
    "Robert Redford",
    "Tim Roth",
    "William Hurt",
    "Winona Ryder",
    "Dermot Mulroney",
    "Donald Crisp",
    "Dwayne Johnson",
    "Emma Thompson",
    "Jack Black",
    "James Caan",
    "Kurt Russell",
    "Maggie Smith",
    "Michael Douglas",
    "Michel Piccoli",
    "Penélope Cruz",
    "Rosario Dawson",
    "Sandra Bullock",
    "Vanessa Redgrave",
    "Vincent Cassel",
    "Angelina Jolie",
    "Arnold Schwarzenegger",
    "Boris Karloff",
    "Charlotte Rampling",
    "Donald Pleasence",
    "Joaquin Phoenix",
    "Joseph Cotten",
    "Juliette Binoche",
    "Kathy Bates",
    "Kirsten Dunst",
    "Mark Ruffalo",
    "Mark Strong",
    "Orson Welles",
    "Paul Rudd",
    "Peter Lorre",
    "Reese Witherspoon",
    "Rod Steiger",
    "Russell Crowe",
    "Steve Martin",
    "Whoopi Goldberg",
    "Akshay Kumar",
    "Alain Delon",
    "Anne Bancroft",
    "Anne Hathaway",
    "Ben Stiller",
    "Burt Reynolds"
]

top_actor_count = (
    df_actors[df_actors["name"].isin(top_actors)]
    .groupby("id")
    .size()
    .reset_index(name="top_actor_count")
)

actors_features = actors_features.merge(
    top_actor_count,
    on="id",
    how="left"
)

actors_features["top_actor_count"] = (
    actors_features["top_actor_count"]
    .fillna(0)
    .astype(int)
)

print(actors_features.head())

        id  actor_count  unique_role_count  top_actor_count
0  1000001          167                 47                3
1  1000002           49                 43                0
2  1000003           36                 31                0
3  1000004           75                 71                1
4  1000005          193                 99                1


Left join

In [ ]:
df_final = df_final.merge(
    actors_features,
    on="id",
    how="left"
)

In [ ]:
df_final["top_actor_count"] = df_final["top_actor_count"].fillna(0).astype(int)

Sonra kontrol:

In [ ]:
new_columns = [
    "actor_count",
    "unique_role_count"
]

print("Silinecek film sayısı:",
      df_final[new_columns].isna().any(axis=1).sum())

df_final = df_final.dropna(subset=new_columns).reset_index(drop=True)

print("Yeni boyut:", df_final.shape)

Silinecek film sayısı: 1623
Yeni boyut: (74623, 138)


In [ ]:
new_columns = [
    "actor_count",
    "unique_role_count"
]

print(df_final[new_columns].isna().sum())

print("Silinecek film sayısı:",
      df_final[new_columns].isna().any(axis=1).sum())

df_final = df_final.dropna(subset=new_columns).reset_index(drop=True)

print(df_final.shape)

actor_count          0
unique_role_count    0
dtype: int64
Silinecek film sayısı: 0
(74623, 138)


HEPSİNİ BİLEŞTİRMEK

In [ ]:
df_final = df_final.drop(columns=["days_theatrical_to_digital_y"], errors="ignore")

In [ ]:
print(df_final.head())
print(df_final.shape)

# NaN olan sütunları ve sayıları
nan_counts = df_final.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

print("\nNaN bulunan sütunlar:")
print(nan_counts)

print("\nToplam NaN sayısı:", nan_counts.sum())


        id                               name    date  minute  rating  Arabic  \
0  1000001                             Barbie  2023.0   114.0    3.86     0.0   
1  1000002                           Parasite  2019.0   133.0    4.56     0.0   
2  1000003  Everything Everywhere All at Once  2022.0   140.0    4.30     0.0   
3  1000004                         Fight Club  1999.0   139.0    4.27     0.0   
4  1000005                         La La Land  2016.0   129.0    4.09     0.0   

   Cantonese  Chinese  Czech  Danish  ...  South Korea  USSR  Argentina  \
0        0.0      0.0    0.0     0.0  ...          0.0   0.0        0.0   
1        0.0      0.0    0.0     0.0  ...          1.0   0.0        0.0   
2        1.0      1.0    0.0     0.0  ...          0.0   0.0        0.0   
3        0.0      0.0    0.0     0.0  ...          0.0   0.0        0.0   
4        0.0      0.0    0.0     0.0  ...          0.0   0.0        0.0   

   Sweden  Australia  Philippines  Hong Kong  actor_count  uni

In [ ]:
print("=" * 50)
print("DATASET CHECK")
print("=" * 50)

print("Shape:", df_final.shape)
print("Duplicate ids:", df_final["id"].duplicated().sum())
print("Duplicate columns:", df_final.columns.duplicated().sum())

xy_cols = [c for c in df_final.columns if c.endswith("_x") or c.endswith("_y")]
print("Remaining _x/_y columns:", xy_cols)

print("Total NaN:", df_final.isna().sum().sum())
print(df_final.info())

DATASET CHECK
Shape: (74623, 138)
Duplicate ids: 0
Duplicate columns: 0
Remaining _x/_y columns: []
Total NaN: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74623 entries, 0 to 74622
Columns: 138 entries, id to top_actor_count
dtypes: float64(133), int64(1), object(4)
memory usage: 78.6+ MB
None


In [ ]:
df_final.to_csv("powerbi_data.csv", index=False)

In [ ]:
from google.colab import files

files.download("powerbi_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>